# LLM Evaluation

**Evaluates** intent classification and attribute extraction prompts against a labelled test dataset.

Metrics produced:
- Per-class Precision / Recall / F1 + macro & weighted averages
- Confusion matrix (heatmap)
- Per-field extraction accuracy
- Hallucination & invalid-value detection
- Latency (mean, p50, p95) and estimated API cost

In [ ]:
import json
import sys
import time
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import seaborn as sns

sys.path.insert(0, "main")          

import LLM_agent as _agent

INT_MODEL = "gpt-4o"
ATT_MODEL = "claude-sonnet-4-5"

VALID_INTENTS = {"RECOMMEND", "ALTERNATIVE", "FEEDBACK_POS", "FEEDBACK_NEG", "SMALLTALK", "OTHER"}

VALID_ATTRIBUTES = {
    "UserAge":        {"young", "adult", "senior"},
    "UserGender":     {"male", "female"},
    "HouseholdType":  {"single", "couple", "family"},
    "TimeOfDay":      {"morning", "afternoon", "night"},
    "DayType":        {"weekday", "weekend"},
    "ProgramType":    {"movie", "series", "news", "documentary", "entertainment"},
    "ProgramGenre":   {"comedy", "drama", "horror", "romance", "news", "documentary",
                       "entertainment", "action", "thriller", "sci-fi", "fantasy"},
    "ProgramDuration":{"short", "medium", "long"},
}

print(f"✅ Setup complete  (model: {INT_MODEL}, {ATT_MODEL})")

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# OpenAI (intent classification — gpt-4o)
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# MasOrange (attribute extraction — claude-opus-4-6)
mo_client = OpenAI(
    api_key=os.getenv("MO_API_KEY"),
    base_url="https://llm.tools.cloud.masorange.es",
)

print("✅ Clients ready")

### Test Datasets

In [ ]:
# Each tuple: (message, expected_intent, difficulty, notes)
INTENT_TEST_CASES = [
    # ── ALTERNATIVE ──────────────────────────────────────────────────────────────
    ('Nada de terror', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('No quiero ver comedias', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Algo que no sea romántico', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Prefiero otro género', 'ALTERNATIVE', 'medium', 'vague genre alternative'),
    ('Sin drama por favor', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Nada de ciencia ficción', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Dame otra opción', 'ALTERNATIVE', 'easy', 'simple alternative'),
    ('Otra cosa', 'ALTERNATIVE', 'easy', 'simple alternative'),
    ('Algo diferente', 'ALTERNATIVE', 'easy', 'simple alternative'),
    ('¿Hay algo más?', 'ALTERNATIVE', 'medium', 'implicit alternative'),
    ('No es lo que busco', 'ALTERNATIVE', 'hard', 'implicit rejection'),
    ('Prefiero ver otra cosa', 'ALTERNATIVE', 'medium', 'explicit preference for alternative'),
    ('Prefiero algo diferente', 'ALTERNATIVE', 'easy', 'explicit preference for alternative'),
    ('¿No tienes otra cosa?', 'ALTERNATIVE', 'medium', 'implicit request for alternative'),
    ('Quiero algo más corto', 'ALTERNATIVE', 'hard', 'attribute-based alternative'),
    ('No quiero ver nada de eso', 'ALTERNATIVE', 'hard', 'vague broad rejection'),
    ('Pon otra cosa', 'ALTERNATIVE', 'easy', 'simple alternative'),
    ('No me apetece eso', 'ALTERNATIVE', 'medium', 'implicit alternative'),
    ('Algo sin violencia', 'ALTERNATIVE', 'easy', 'content-based alternative'),
    ('Prefiero algo más tranquilo', 'ALTERNATIVE', 'medium', 'mood-based alternative'),
    ('Sin películas de autor', 'ALTERNATIVE', 'medium', 'style-based alternative'),
    ('Algo más moderno', 'ALTERNATIVE', 'medium', 'attribute-based alternative'),
    ('No quiero nada infantil', 'ALTERNATIVE', 'easy', 'audience-based alternative'),
    ('Algo que no sea tan largo', 'ALTERNATIVE', 'medium', 'duration-based alternative'),
    ('Prefiero una serie', 'ALTERNATIVE', 'medium', 'type-based alternative'),
    ('Nada de documentales', 'ALTERNATIVE', 'easy', 'type rejection'),
    ('¿Tienes algo más animado?', 'ALTERNATIVE', 'medium', 'mood alternative'),
    ('Algo con más acción', 'ALTERNATIVE', 'hard', 'attribute alternative, could be RECOMMEND'),
    ('No, algo distinto', 'ALTERNATIVE', 'easy', 'simple alternative'),
    ('Prefiero ver algo más ligero', 'ALTERNATIVE', 'medium', 'mood-based alternative'),
    ('No quiero nada de romance', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Sin animación', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Nada de reality', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('No, cambia de género', 'ALTERNATIVE', 'medium', 'genre change request'),
    ('Algo más serio', 'ALTERNATIVE', 'medium', 'mood-based alternative'),
    ('Prefiero algo con más intriga', 'ALTERNATIVE', 'medium', 'mood-based alternative'),
    ('Nada de musicales', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Algo sin subtítulos', 'ALTERNATIVE', 'medium', 'format-based alternative'),
    ('¿Tienes otra opción?', 'ALTERNATIVE', 'easy', 'explicit alternative request'),
    ('Quiero ver otro tipo de contenido', 'ALTERNATIVE', 'medium', 'type change'),
    ('No me van las series largas', 'ALTERNATIVE', 'medium', 'duration rejection'),
    ('Prefiero algo más clásico', 'ALTERNATIVE', 'medium', 'style-based alternative'),
    ('Sin series de época', 'ALTERNATIVE', 'medium', 'subgenre rejection'),
    ('Algo menos dramático', 'ALTERNATIVE', 'medium', 'mood-based alternative'),
    ('No me gustan los thrillers', 'ALTERNATIVE', 'easy', 'genre rejection → ALTERNATIVE per rules'),
    ('Prefiero una película en vez de serie', 'ALTERNATIVE', 'medium', 'type-based alternative'),
    ('No quiero nada oscuro', 'ALTERNATIVE', 'medium', 'tone-based alternative'),
    ('Dame algo más antiguo', 'ALTERNATIVE', 'medium', 'era-based alternative'),
    ('Sin fantasía', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Algo diferente al terror', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Prefiero algo con más humor', 'ALTERNATIVE', 'medium', 'tone-based alternative'),
    ('Sin cosas de superhéroes', 'ALTERNATIVE', 'easy', 'subgenre rejection'),
    ('No quiero drama familiar', 'ALTERNATIVE', 'medium', 'subgenre rejection'),
    ('Algo con menos violencia', 'ALTERNATIVE', 'medium', 'content-based alternative'),
    ('Prefiero ver algo en español', 'ALTERNATIVE', 'medium', 'language-based alternative'),
    ('Nada de animé', 'ALTERNATIVE', 'easy', 'subgenre rejection'),
    ('Algo que no sea tan denso', 'ALTERNATIVE', 'medium', 'complexity-based alternative'),
    ('Prefiero algo más reciente', 'ALTERNATIVE', 'medium', 'era-based alternative'),
    ('No, busca otra cosa', 'ALTERNATIVE', 'easy', 'simple alternative'),
    ('Otro género, por favor', 'ALTERNATIVE', 'easy', 'genre change'),
    ('Algo que se entienda mejor', 'ALTERNATIVE', 'medium', 'complexity-based alternative'),
    ('Sin escenas fuertes', 'ALTERNATIVE', 'medium', 'content-based alternative'),
    ('No quiero nada de crimen', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Prefiero algo más conocido', 'ALTERNATIVE', 'medium', 'popularity-based alternative'),
    ('Nada de western', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Dame algo diferente', 'ALTERNATIVE', 'easy', 'simple alternative'),
    ('Algo sin tanto diálogo', 'ALTERNATIVE', 'medium', 'style-based alternative'),
    ('Prefiero algo más emocionante', 'ALTERNATIVE', 'medium', 'mood-based alternative'),
    ('No quiero nada muy triste', 'ALTERNATIVE', 'medium', 'tone-based alternative'),
    ('Sin dramas históricos', 'ALTERNATIVE', 'medium', 'subgenre rejection'),
    ('Algo que pueda ver con mi madre', 'ALTERNATIVE', 'medium', 'audience-based alternative'),
    ('Prefiero otra serie', 'ALTERNATIVE', 'easy', 'type-based alternative'),
    ('Nada de deportes', 'ALTERNATIVE', 'easy', 'genre rejection'),
    ('Algo sin gore', 'ALTERNATIVE', 'medium', 'content-based alternative'),
    ('No me apetece ver eso ahora mismo', 'ALTERNATIVE', 'medium', 'implicit alternative'),
    ('Quiero algo menos intenso', 'ALTERNATIVE', 'medium', 'mood-based alternative'),
    ('Dame una opción distinta', 'ALTERNATIVE', 'easy', 'simple alternative'),
    ('Sin comedias románticas', 'ALTERNATIVE', 'easy', 'subgenre rejection'),
    ('No quiero nada extranjero', 'ALTERNATIVE', 'medium', 'language-based alternative'),
    ('Algo más familiar', 'ALTERNATIVE', 'medium', 'audience-based alternative'),

    # ── FEEDBACK_NEG ──────────────────────────────────────────────────────────────
    ('No me gusta el drama', 'FEEDBACK_NEG', 'easy', 'genre dislike'),
    ('No me gustan las comedias', 'FEEDBACK_NEG', 'medium', 'genre dislike plural'),
    ('No me gusta ese tipo de películas', 'FEEDBACK_NEG', 'medium', 'type dislike'),
    ('Esa no me gusta', 'FEEDBACK_NEG', 'easy', 'specific recommendation rejection'),
    ('Ya la vi', 'FEEDBACK_NEG', 'easy', 'already seen'),
    ('No me convence esa película', 'FEEDBACK_NEG', 'easy', 'specific rejection'),
    ('Esa es muy larga', 'FEEDBACK_NEG', 'medium', 'attribute-based rejection'),
    ('No, esa no', 'FEEDBACK_NEG', 'medium', 'short negative'),
    ('La he visto ya', 'FEEDBACK_NEG', 'easy', 'already seen variant'),
    ('Esa película no me llama la atención', 'FEEDBACK_NEG', 'medium', 'paraphrase'),
    ('Me aburre un poco', 'FEEDBACK_NEG', 'hard', 'implicit mild rejection'),
    ('Eso no me mola', 'FEEDBACK_NEG', 'easy', 'colloquial rejection'),
    ('Qué aburrida', 'FEEDBACK_NEG', 'easy', 'direct negative reaction'),
    ('No me interesa', 'FEEDBACK_NEG', 'easy', 'disinterest'),
    ('Qué rollo', 'FEEDBACK_NEG', 'easy', 'colloquial rejection'),
    ('Ese actor no me gusta', 'FEEDBACK_NEG', 'medium', 'actor-based dislike'),
    ('Demasiado violenta', 'FEEDBACK_NEG', 'medium', 'content-based rejection'),
    ('No, gracias', 'FEEDBACK_NEG', 'easy', 'polite rejection'),
    ('Paso de eso', 'FEEDBACK_NEG', 'easy', 'colloquial rejection'),
    ('No es para mí', 'FEEDBACK_NEG', 'medium', 'personal incompatibility'),
    ('Ya la he visto tres veces', 'FEEDBACK_NEG', 'easy', 'already seen emphatic'),
    ('No me convence', 'FEEDBACK_NEG', 'easy', 'general rejection'),
    ('Esa serie no me gusta nada', 'FEEDBACK_NEG', 'easy', 'emphatic series rejection'),
    ('No, esa no me interesa', 'FEEDBACK_NEG', 'easy', 'specific rejection'),
    ('Ya la había visto', 'FEEDBACK_NEG', 'easy', 'already seen'),
    ('No es de mi estilo', 'FEEDBACK_NEG', 'medium', 'personal incompatibility'),
    ('Esa serie me parece malísima', 'FEEDBACK_NEG', 'easy', 'emphatic rejection'),
    ('No me gusta el director', 'FEEDBACK_NEG', 'medium', 'creative rejection'),
    ('Ya vi esa película el año pasado', 'FEEDBACK_NEG', 'easy', 'already seen with detail'),
    ('Demasiado larga para hoy', 'FEEDBACK_NEG', 'medium', 'duration rejection'),
    ('No me llama nada', 'FEEDBACK_NEG', 'easy', 'disinterest'),
    ('Ese actor me cae mal', 'FEEDBACK_NEG', 'medium', 'cast-based rejection'),
    ('No, para nada', 'FEEDBACK_NEG', 'easy', 'emphatic rejection'),
    ('Esa ya la tengo vista', 'FEEDBACK_NEG', 'easy', 'already seen colloquial'),
    ('No es lo que esperaba', 'FEEDBACK_NEG', 'medium', 'expectation mismatch'),
    ('Me parece un rollo', 'FEEDBACK_NEG', 'easy', 'colloquial rejection'),
    ('Muy lenta para mí', 'FEEDBACK_NEG', 'medium', 'pace-based rejection'),
    ('Esa serie no aguanto ni el primer capítulo', 'FEEDBACK_NEG', 'hard', 'emphatic already-seen'),
    ('No me convence el argumento', 'FEEDBACK_NEG', 'medium', 'plot-based rejection'),
    ('Esa la vi hace poco', 'FEEDBACK_NEG', 'easy', 'recently seen'),
    ('Qué mala pinta tiene', 'FEEDBACK_NEG', 'easy', 'negative impression'),
    ('Buf, no', 'FEEDBACK_NEG', 'easy', 'colloquial rejection'),
    ('No me gusta esa actriz', 'FEEDBACK_NEG', 'medium', 'cast-based rejection'),
    ('Demasiado vieja esa película', 'FEEDBACK_NEG', 'medium', 'era-based rejection'),
    ('Eso no es para mí', 'FEEDBACK_NEG', 'medium', 'personal incompatibility'),
    ('No quiero ver esa', 'FEEDBACK_NEG', 'easy', 'direct rejection'),
    ('Me aburre solo de verla', 'FEEDBACK_NEG', 'medium', 'implicit rejection'),
    ('Pasa, no me gusta', 'FEEDBACK_NEG', 'easy', 'colloquial rejection'),
    ('La vi con mi ex, malos recuerdos', 'FEEDBACK_NEG', 'hard', 'personal reason rejection'),
    ('No me gusta ese tipo de finales', 'FEEDBACK_NEG', 'hard', 'plot-based rejection'),
    ('Demasiado comercial', 'FEEDBACK_NEG', 'hard', 'style rejection'),
    ('No es lo que busco ahora', 'FEEDBACK_NEG', 'medium', 'situational rejection'),
    ('Esa ya no me apetece', 'FEEDBACK_NEG', 'medium', 'changed preference rejection'),
    ('Meh', 'FEEDBACK_NEG', 'hard', 'minimal negative'),
    ('No me entusiasma', 'FEEDBACK_NEG', 'medium', 'mild rejection'),
    ('Es demasiado infantil para mí', 'FEEDBACK_NEG', 'medium', 'audience mismatch'),
    ('Ese programa es un bodrio', 'FEEDBACK_NEG', 'easy', 'emphatic rejection'),
    ('No, gracias, esa no', 'FEEDBACK_NEG', 'easy', 'polite specific rejection'),
    ('Qué asco de película', 'FEEDBACK_NEG', 'easy', 'strong rejection'),
    ('No me va ese rollo', 'FEEDBACK_NEG', 'easy', 'colloquial rejection'),
    ('Ya estoy harta de esa serie', 'FEEDBACK_NEG', 'medium', 'saturation rejection'),
    ('La empecé y la dejé a medias', 'FEEDBACK_NEG', 'hard', 'abandonment signal'),
    ('Ni hablar de esa', 'FEEDBACK_NEG', 'easy', 'emphatic rejection'),

    # ── FEEDBACK_POS ──────────────────────────────────────────────────────────────
    ('Me gusta', 'FEEDBACK_POS', 'easy', 'simple positive'),
    ('Perfecto', 'FEEDBACK_POS', 'easy', 'simple positive'),
    ('La veo', 'FEEDBACK_POS', 'easy', 'commitment'),
    ('De acuerdo', 'FEEDBACK_POS', 'easy', 'agreement'),
    ('Buena idea', 'FEEDBACK_POS', 'medium', 'implicit positive'),
    ('Sí, me apetece', 'FEEDBACK_POS', 'medium', 'positive with desire'),
    ('Esa me parece bien', 'FEEDBACK_POS', 'medium', 'acceptance'),
    ('Me gusta ese tipo de series', 'FEEDBACK_POS', 'medium', 'positive about type/genre'),
    ('Sí, la veo', 'FEEDBACK_POS', 'easy', 'explicit acceptance'),
    ('Genial', 'FEEDBACK_POS', 'easy', 'enthusiastic positive'),
    ('Qué buena pinta tiene', 'FEEDBACK_POS', 'easy', 'positive impression'),
    ('Sí, ponla', 'FEEDBACK_POS', 'easy', 'explicit command to play'),
    ('Me encanta ese género', 'FEEDBACK_POS', 'easy', 'genre affinity'),
    ('Perfecto, esa la veo', 'FEEDBACK_POS', 'easy', 'emphatic acceptance'),
    ('¡Sí!', 'FEEDBACK_POS', 'easy', 'emphatic affirmation'),
    ('Suena bien', 'FEEDBACK_POS', 'medium', 'implicit positive'),
    ('Eso me gusta', 'FEEDBACK_POS', 'easy', 'direct positive'),
    ('Me parece una buena elección', 'FEEDBACK_POS', 'medium', 'formal acceptance'),
    ('La pondré', 'FEEDBACK_POS', 'easy', 'future commitment'),
    ('Sí, me parece bien', 'FEEDBACK_POS', 'easy', 'simple agreement'),
    ('Esa mola', 'FEEDBACK_POS', 'easy', 'colloquial positive'),
    ('Perfecto, muchas gracias', 'FEEDBACK_POS', 'medium', 'positive with thanks'),
    ('Eso me parece genial', 'FEEDBACK_POS', 'easy', 'enthusiastic acceptance'),
    ('Sí, esa me apetece mucho', 'FEEDBACK_POS', 'easy', 'enthusiastic agreement'),
    ('Me encanta, la pongo ya', 'FEEDBACK_POS', 'easy', 'enthusiastic with action'),
    ('Buena elección', 'FEEDBACK_POS', 'easy', 'compliment to recommendation'),
    ('Qué buena idea', 'FEEDBACK_POS', 'easy', 'positive reaction'),
    ('Sí, ponla', 'FEEDBACK_POS', 'easy', 'command to play'),
    ('Vale, la veo', 'FEEDBACK_POS', 'easy', 'casual acceptance'),
    ('Perfecto, justo lo que quería', 'FEEDBACK_POS', 'easy', 'emphatic acceptance'),
    ('Esa me llama mucho la atención', 'FEEDBACK_POS', 'medium', 'positive interest'),
    ('Tiene buena pinta', 'FEEDBACK_POS', 'easy', 'positive impression'),
    ('Sí, esa está bien', 'FEEDBACK_POS', 'easy', 'casual positive'),
    ('Me parece una muy buena opción', 'FEEDBACK_POS', 'easy', 'formal acceptance'),
    ('¡Qué buena!', 'FEEDBACK_POS', 'easy', 'enthusiastic positive'),
    ('Ese me encanta', 'FEEDBACK_POS', 'easy', 'direct positive'),
    ('Estupendo', 'FEEDBACK_POS', 'easy', 'formal positive'),
    ('Excelente elección', 'FEEDBACK_POS', 'easy', 'formal compliment'),
    ('Sí, eso me apetece', 'FEEDBACK_POS', 'easy', 'positive desire'),
    ('La pondré esta noche', 'FEEDBACK_POS', 'easy', 'future commitment'),
    ('De acuerdo, esa me gusta', 'FEEDBACK_POS', 'easy', 'agreement with positive'),
    ('Sí, me parece perfecta', 'FEEDBACK_POS', 'easy', 'emphatic agreement'),
    ('Esa me la apunto', 'FEEDBACK_POS', 'medium', 'future intent positive'),
    ('Venga, la veo', 'FEEDBACK_POS', 'easy', 'casual acceptance colloquial'),
    ('Muy buena sugerencia', 'FEEDBACK_POS', 'medium', 'formal positive'),
    ('Sí, me convence', 'FEEDBACK_POS', 'easy', 'acceptance'),
    ('Genial, justo lo que necesitaba', 'FEEDBACK_POS', 'easy', 'emphatic acceptance'),
    ('Perfecto, muchas gracias', 'FEEDBACK_POS', 'easy', 'positive with thanks'),
    ('Sí, eso está fenomenal', 'FEEDBACK_POS', 'easy', 'emphatic positive'),
    ('Me parece bien esa opción', 'FEEDBACK_POS', 'easy', 'neutral-positive acceptance'),
    ('¡Me encanta!', 'FEEDBACK_POS', 'easy', 'enthusiastic positive'),
    ('Sí, claro', 'FEEDBACK_POS', 'medium', 'brief agreement'),

    # ── RECOMMEND ──────────────────────────────────────────────────────────────
    ('Quiero ver una película', 'RECOMMEND', 'easy', 'type request'),
    ('Qué puedo ver', 'RECOMMEND', 'easy', 'open recommendation'),
    ('Recomiéndame algo', 'RECOMMEND', 'easy', 'explicit request'),
    ('Quiero ver una comedia', 'RECOMMEND', 'easy', 'genre + type'),
    ('Ponme algo de acción', 'RECOMMEND', 'easy', 'genre request'),
    ('¿Qué hay bueno hoy?', 'RECOMMEND', 'medium', 'temporal rec request'),
    ('Busco una serie entretenida', 'RECOMMEND', 'medium', 'series request'),
    ('¿Qué series tienes?', 'RECOMMEND', 'easy', 'catalog inquiry'),
    ('Dame algo para ver esta noche', 'RECOMMEND', 'medium', 'temporal request'),
    ('Necesito una recomendación', 'RECOMMEND', 'easy', 'explicit request variant'),
    ('Qué veo ahora', 'RECOMMEND', 'easy', 'immediate request'),
    ('Una peli de risas', 'RECOMMEND', 'easy', 'colloquial genre request'),
    ('Algo para toda la familia', 'RECOMMEND', 'medium', 'audience request'),
    ('¿Tienes algo de terror?', 'RECOMMEND', 'easy', 'genre catalog inquiry'),
    ('Pon algo relajante', 'RECOMMEND', 'medium', 'mood-based request'),
    ('Una serie corta', 'RECOMMEND', 'medium', 'duration-type request'),
    ('¿Hay alguna serie nueva?', 'RECOMMEND', 'medium', 'novelty request'),
    ('Algo para los niños', 'RECOMMEND', 'medium', 'audience-based request'),
    ('Dame una película de aventuras', 'RECOMMEND', 'easy', 'explicit genre+type request'),
    ('¿Qué puedo ver esta tarde?', 'RECOMMEND', 'medium', 'temporal open request'),
    ('Busco algo de suspense', 'RECOMMEND', 'easy', 'genre request'),
    ('Quiero ver un documental', 'RECOMMEND', 'easy', 'type request'),
    ('Recomiéndame una serie', 'RECOMMEND', 'easy', 'explicit series request'),
    ('Quiero ver algo de humor', 'RECOMMEND', 'easy', 'mood-based request'),
    ('Dame algo de fantasía', 'RECOMMEND', 'easy', 'genre request'),
    ('Quiero una peli de miedo', 'RECOMMEND', 'easy', 'colloquial genre request'),
    ('Pon algo de romance', 'RECOMMEND', 'easy', 'genre request'),
    ('¿Qué series hay disponibles?', 'RECOMMEND', 'easy', 'catalog inquiry'),
    ('Busco algo con buenos actores', 'RECOMMEND', 'medium', 'quality-based request'),
    ('Quiero ver algo entretenido', 'RECOMMEND', 'easy', 'mood request'),
    ('Dame una recomendación', 'RECOMMEND', 'easy', 'explicit request'),
    ('¿Qué me pones?', 'RECOMMEND', 'easy', 'open request colloquial'),
    ('Algo para pasar el rato', 'RECOMMEND', 'easy', 'casual request'),
    ('¿Tienes algo de animación?', 'RECOMMEND', 'easy', 'genre catalog inquiry'),
    ('Quiero ver algo en familia esta tarde', 'RECOMMEND', 'medium', 'audience + time request'),
    ('Busco una película de risa', 'RECOMMEND', 'easy', 'mood-based request'),
    ('Algo para ver el domingo', 'RECOMMEND', 'medium', 'temporal request'),
    ('Pon lo que sea', 'RECOMMEND', 'hard', 'completely open request'),
    ('Quiero ver algo intenso', 'RECOMMEND', 'medium', 'mood-based request'),
    ('Dame algo para dos horas', 'RECOMMEND', 'medium', 'duration-based request'),
    ('¿Tienes thrillers?', 'RECOMMEND', 'easy', 'genre catalog inquiry'),
    ('Quiero una comedia ligera', 'RECOMMEND', 'easy', 'genre+mood request'),
    ('Algo para antes de cenar', 'RECOMMEND', 'medium', 'temporal request'),
    ('Una buena película', 'RECOMMEND', 'medium', 'quality request'),
    ('Dame algo de drama', 'RECOMMEND', 'easy', 'genre request'),
    ('Quiero ver algo de acción americana', 'RECOMMEND', 'medium', 'genre+origin request'),
    ('¿Qué hay de nuevo?', 'RECOMMEND', 'medium', 'novelty request'),
    ('Algo para ver tranquilamente', 'RECOMMEND', 'medium', 'mood request'),
    ('Quiero una serie de misterio', 'RECOMMEND', 'easy', 'genre + type request'),
    ('Dame algo bien valorado', 'RECOMMEND', 'medium', 'quality-based request'),
    ('¿Qué puedo ver con los niños?', 'RECOMMEND', 'medium', 'audience request'),
    ('Una peli de ciencia ficción buena', 'RECOMMEND', 'easy', 'genre + quality request'),

    # ── SMALLTALK ──────────────────────────────────────────────────────────────
    ('Hola', 'SMALLTALK', 'easy', 'greeting'),
    ('Gracias', 'SMALLTALK', 'easy', 'thanks'),
    ('Adiós', 'SMALLTALK', 'easy', 'farewell'),
    ('¿Cómo estás?', 'SMALLTALK', 'medium', 'social question'),
    ('Muy bien', 'SMALLTALK', 'hard', 'ambiguous positive'),
    ('Buenas tardes', 'SMALLTALK', 'easy', 'greeting'),
    ('¿Qué tal?', 'SMALLTALK', 'easy', 'casual greeting'),
    ('Hasta luego', 'SMALLTALK', 'easy', 'farewell variant'),
    ('Muchas gracias', 'SMALLTALK', 'easy', 'emphatic thanks'),
    ('¿Qué eres?', 'SMALLTALK', 'medium', 'identity question'),
    ('Buen trabajo', 'SMALLTALK', 'medium', 'compliment'),
    ('Buenos días', 'SMALLTALK', 'easy', 'morning greeting'),
    ('Hasta pronto', 'SMALLTALK', 'easy', 'farewell variant'),
    ('¿Quién eres?', 'SMALLTALK', 'medium', 'identity question variant'),
    ('¡Hola!', 'SMALLTALK', 'easy', 'greeting with exclamation'),
    ('Eres muy útil', 'SMALLTALK', 'medium', 'compliment'),
    ('Oye, hola', 'SMALLTALK', 'easy', 'casual greeting'),
    ('Hasta mañana', 'SMALLTALK', 'easy', 'farewell variant'),
    ('Gracias por tu ayuda', 'SMALLTALK', 'easy', 'thanks with detail'),
    ('No te preocupes', 'SMALLTALK', 'medium', 'reassurance'),
    ('Vale, gracias', 'SMALLTALK', 'medium', 'brief thanks'),
    ('Muy bien, gracias', 'SMALLTALK', 'hard', 'ambiguous — could be FEEDBACK_POS'),
    ('¿Hablas español?', 'SMALLTALK', 'medium', 'identity/capability question'),
    ('¿Puedes ayudarme?', 'SMALLTALK', 'medium', 'open help request'),
    ('Qué listo eres', 'SMALLTALK', 'medium', 'compliment'),
    ('Nos vemos', 'SMALLTALK', 'easy', 'farewell'),
    ('¡Hasta luego!', 'SMALLTALK', 'easy', 'farewell with exclamation'),
    ('Eres una IA, ¿no?', 'SMALLTALK', 'medium', 'identity question'),
    ('Qué amable', 'SMALLTALK', 'medium', 'compliment'),
    ('No, nada más', 'SMALLTALK', 'hard', 'closing statement — ambiguous'),
    ('Por hoy es suficiente', 'SMALLTALK', 'hard', 'closing statement'),
    ('Estoy aburrido', 'SMALLTALK', 'hard', 'state description — could be RECOMMEND'),
    ('Llevas razón', 'SMALLTALK', 'medium', 'agreement/compliment'),
    ('Buenas noches', 'SMALLTALK', 'easy', 'farewell/greeting'),

    # ── OTHER ──────────────────────────────────────────────────────────────
    ('¿Cuánto cuesta la suscripción?', 'OTHER', 'easy', 'billing question'),
    ('¿Cómo me doy de baja?', 'OTHER', 'easy', 'cancellation question'),
    ('¿Puedo ver en otro idioma?', 'OTHER', 'medium', 'language/settings question'),
    ('Mi televisor no funciona', 'OTHER', 'easy', 'technical support'),
    ('¿Cuántos dispositivos puedo usar?', 'OTHER', 'easy', 'account question'),
    ('¿Tenéis subtítulos?', 'OTHER', 'easy', 'accessibility question'),
    ('¿Cómo cambio la calidad del vídeo?', 'OTHER', 'medium', 'settings question'),
    ('Se me ha olvidado la contraseña', 'OTHER', 'easy', 'account recovery'),
    ('¿Hay una app para el móvil?', 'OTHER', 'easy', 'product question'),
    ('¿Puedo descargar capítulos?', 'OTHER', 'easy', 'feature question'),
    ('¿Cuándo se estrena la nueva temporada?', 'OTHER', 'medium', 'release date question'),
    ('¿Puedo ver en 4K?', 'OTHER', 'easy', 'quality/settings question'),
    ('¿Cómo cancelo mi cuenta?', 'OTHER', 'easy', 'account management'),
    ('Quiero cambiar mi contraseña', 'OTHER', 'easy', 'account management'),
    ('¿Hay versión en doblaje?', 'OTHER', 'medium', 'language/localization question'),
    ('¿Funciona en smart TV?', 'OTHER', 'easy', 'compatibility question'),
    ('¿Cuánto dura la suscripción?', 'OTHER', 'easy', 'billing question'),
    ('No carga el vídeo', 'OTHER', 'easy', 'technical support'),
    ('La imagen se congela', 'OTHER', 'easy', 'technical support'),

]

# Each tuple: (message, expected_attrs_dict)
EXTRACTION_TEST_CASES = [
    ('Quiero ver una comedia de 90 minutos',
        {'ProgramType': 'movie', 'ProgramGenre': 'comedy', 'ProgramDuration': 'long'}),
    ('Soy un chico de 32 años',
        {'UserAge': 'young', 'UserGender': 'male'}),
    ('Somos una familia y queremos ver una serie esta tarde',
        {'HouseholdType': 'family', 'ProgramType': 'series', 'TimeOfDay': 'afternoon'}),
    ('Quiero ver algo de terror por la noche',
        {'ProgramGenre': 'horror', 'TimeOfDay': 'night'}),
    ('Soy una señora mayor de 70 años y vivo sola',
        {'UserAge': 'senior', 'UserGender': 'female', 'HouseholdType': 'single'}),
    ('Un documental corto para el fin de semana',
        {'ProgramType': 'documentary', 'ProgramGenre': 'documentary', 'ProgramDuration': 'short', 'DayType': 'weekend'}),
    ('Algo de acción, somos una pareja',
        {'ProgramGenre': 'action', 'HouseholdType': 'couple'}),
    ('Quiero ver las noticias',
        {'ProgramType': 'news', 'ProgramGenre': 'news'}),
    ('Una película de ciencia ficción larga',
        {'ProgramType': 'movie', 'ProgramGenre': 'sci-fi', 'ProgramDuration': 'long'}),
    ('Hola, ¿qué hay?',
        {}),
    ('Algo entretenido para esta noche entre semana',
        {'ProgramGenre': 'entertainment', 'TimeOfDay': 'night', 'DayType': 'weekday'}),
    ('Tengo 45 años y vivo en pareja',
        {'UserAge': 'adult', 'HouseholdType': 'couple'}),
    ('Tengo 15 años y me gustan las series de fantasía',
        {'UserAge': 'young', 'ProgramType': 'series', 'ProgramGenre': 'fantasy'}),
    ('Soy una mujer de unos 55 años',
        {'UserAge': 'adult', 'UserGender': 'female'}),
    ('Quiero ver algo corto antes de dormir',
        {'ProgramDuration': 'short', 'TimeOfDay': 'night'}),
    ('Somos dos amigos viendo algo el sábado',
        {'HouseholdType': 'couple', 'DayType': 'weekend'}),
    ('Dame un thriller largo',
        {'ProgramGenre': 'thriller', 'ProgramDuration': 'long'}),
    ('Una película romántica para esta tarde',
        {'ProgramType': 'movie', 'ProgramGenre': 'romance', 'TimeOfDay': 'afternoon'}),
    ('Tengo 8 años',
        {'UserAge': 'young'}),
    ('Un señor de 75 años buscando noticias por la mañana',
        {'UserAge': 'senior', 'UserGender': 'male', 'ProgramType': 'news', 'ProgramGenre': 'news', 'TimeOfDay': 'morning'}),
    ('Quiero algo de entretenimiento para el fin de semana',
        {'ProgramGenre': 'entertainment', 'DayType': 'weekend'}),
    ('Somos una familia con niños, algo para el domingo por la tarde',
        {'HouseholdType': 'family', 'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Una comedia corta',
        {'ProgramGenre': 'comedy', 'ProgramDuration': 'short'}),
    ('Tengo 28 años, soy chica y vivo sola',
        {'UserAge': 'young', 'UserGender': 'female', 'HouseholdType': 'single'}),
    ('Un drama largo para el fin de semana',
        {'ProgramGenre': 'drama', 'ProgramDuration': 'long', 'DayType': 'weekend'}),
    ('Algo para ver entre semana por la mañana',
        {'DayType': 'weekday', 'TimeOfDay': 'morning'}),
    ('Mi marido y yo buscamos una película de acción',
        {'HouseholdType': 'couple', 'ProgramType': 'movie', 'ProgramGenre': 'action'}),
    ('Soy jubilado y busco algo tranquilo',
        {'UserAge': 'senior'}),
    ('Un documental largo',
        {'ProgramType': 'documentary', 'ProgramGenre': 'documentary', 'ProgramDuration': 'long'}),
    ('Quiero ver el telediario',
        {'ProgramType': 'news', 'ProgramGenre': 'news'}),
    ('Quiero una serie de ciencia ficción',
        {'ProgramType': 'series', 'ProgramGenre': 'sci-fi'}),
    ('Algo corto para ver entre semana por la tarde',
        {'ProgramDuration': 'short', 'DayType': 'weekday', 'TimeOfDay': 'afternoon'}),
    ('Quiero ver una película esta noche, somos pareja',
        {'ProgramType': 'movie', 'TimeOfDay': 'night', 'HouseholdType': 'couple'}),
    ('Soy un hombre mayor de 68 años',
        {'UserAge': 'senior', 'UserGender': 'male'}),
    ('Recomiéndame algo de humor para el fin de semana',
        {'ProgramGenre': 'comedy', 'DayType': 'weekend'}),
    ('Tengo 40 años, vivo solo y quiero ver algo de suspense esta noche',
        {'UserAge': 'adult', 'HouseholdType': 'single', 'ProgramGenre': 'thriller', 'TimeOfDay': 'night'}),
    ('Una serie de drama mediana',
        {'ProgramType': 'series', 'ProgramGenre': 'drama', 'ProgramDuration': 'medium'}),
    ('Somos una familia y queremos ver las noticias del mediodía',
        {'HouseholdType': 'family', 'ProgramType': 'news', 'ProgramGenre': 'news', 'TimeOfDay': 'afternoon'}),
    ('Tengo 22 años, soy chico',
        {'UserAge': 'young', 'UserGender': 'male'}),
    ('Pon algo de acción corto entre semana',
        {'ProgramGenre': 'action', 'ProgramDuration': 'short', 'DayType': 'weekday'}),
    ('Quiero algo de comedia o entretenimiento para el fin de semana por la tarde',
        {'ProgramGenre': 'comedy', 'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Tengo 20 años',
        {'UserAge': 'young'}),
    ('Soy un chico joven, unos 25',
        {'UserAge': 'young', 'UserGender': 'male'}),
    ('Tengo 60 años',
        {'UserAge': 'senior'}),
    ('Soy una persona de mediana edad',
        {'UserAge': 'adult'}),
    ('Tengo 80 años y vivo sola',
        {'UserAge': 'senior', 'HouseholdType': 'single'}),
    ('Soy jubilada y tengo 67 años',
        {'UserAge': 'senior', 'UserGender': 'female'}),
    ('Tengo 35 años',
        {'UserAge': 'adult'}),
    ('Soy mayor, tengo 72',
        {'UserAge': 'senior'}),
    ('Soy hombre',
        {'UserGender': 'male'}),
    ('Soy mujer',
        {'UserGender': 'female'}),
    ('Soy una chica',
        {'UserGender': 'female'}),
    ('Soy un señor',
        {'UserGender': 'male'}),
    ('Vivo solo',
        {'HouseholdType': 'single'}),
    ('Somos tres en casa',
        {'HouseholdType': 'family'}),
    ('Estoy con mi pareja',
        {'HouseholdType': 'couple'}),
    ('Vivo con mi familia',
        {'HouseholdType': 'family'}),
    ('Estoy con mis hijos',
        {'HouseholdType': 'family'}),
    ('Somos una pareja',
        {'HouseholdType': 'couple'}),
    ('Quiero ver algo mañana',
        {'TimeOfDay': 'morning'}),
    ('Es por la mañana',
        {'TimeOfDay': 'morning'}),
    ('Son las 9 de la mañana',
        {'TimeOfDay': 'morning'}),
    ('Es tarde, ya son las 11 de la noche',
        {'TimeOfDay': 'night'}),
    ('Estamos a domingo',
        {'DayType': 'weekend'}),
    ('Es sábado por la tarde',
        {'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Es un martes',
        {'DayType': 'weekday'}),
    ('Estamos de semana',
        {'DayType': 'weekday'}),
    ('Mediodía del viernes',
        {'DayType': 'weekday', 'TimeOfDay': 'afternoon'}),
    ('Noche del domingo',
        {'DayType': 'weekend', 'TimeOfDay': 'night'}),
    ('Una peli',
        {'ProgramType': 'movie'}),
    ('Una serie',
        {'ProgramType': 'series'}),
    ('El telediario',
        {'ProgramType': 'news', 'ProgramGenre': 'news'}),
    ('Un documental',
        {'ProgramType': 'documentary', 'ProgramGenre': 'documentary'}),
    ('Algo de humor',
        {'ProgramGenre': 'comedy'}),
    ('Algo de terror',
        {'ProgramGenre': 'horror'}),
    ('Una de acción',
        {'ProgramGenre': 'action'}),
    ('Algo romántico',
        {'ProgramGenre': 'romance'}),
    ('Un thriller',
        {'ProgramGenre': 'thriller'}),
    ('Algo de fantasía',
        {'ProgramGenre': 'fantasy'}),
    ('Algo corto',
        {'ProgramDuration': 'short'}),
    ('Algo largo',
        {'ProgramDuration': 'long'}),
    ('Una de mediana duración',
        {'ProgramDuration': 'medium'}),
    ('Un programa de entretenimiento',
        {'ProgramType': 'entertainment', 'ProgramGenre': 'entertainment'}),
    ('Soy un chico de 19 años, quiero ver algo de ciencia ficción esta noche entre semana',
        {'UserAge': 'young', 'UserGender': 'male', 'ProgramGenre': 'sci-fi', 'TimeOfDay': 'night', 'DayType': 'weekday'}),
    ('Mi mujer y yo tenemos 50 años y queremos ver una comedia el sábado por la noche',
        {'UserAge': 'adult', 'HouseholdType': 'couple', 'ProgramGenre': 'comedy', 'DayType': 'weekend', 'TimeOfDay': 'night'}),
    ('Somos una familia con niños pequeños, algo de animación para el domingo',
        {'HouseholdType': 'family', 'ProgramGenre': 'fantasy', 'DayType': 'weekend'}),
    ('Tengo 45 años, soy hombre, vivo solo y quiero un thriller largo para el viernes por la noche',
        {'UserAge': 'adult', 'UserGender': 'male', 'HouseholdType': 'single', 'ProgramGenre': 'thriller', 'ProgramDuration': 'long', 'DayType': 'weekend', 'TimeOfDay': 'night'}),
    ('Una señora de 65 años y su marido quieren ver una película romántica esta tarde',
        {'UserAge': 'senior', 'HouseholdType': 'couple', 'ProgramType': 'movie', 'ProgramGenre': 'romance', 'TimeOfDay': 'afternoon'}),
    ('Tengo 30 años, soy mujer y busco una serie de drama corta para entre semana',
        {'UserAge': 'young', 'UserGender': 'female', 'ProgramType': 'series', 'ProgramGenre': 'drama', 'ProgramDuration': 'short', 'DayType': 'weekday'}),
    ('Somos dos adultos sin hijos, queremos un documental largo el sábado',
        {'HouseholdType': 'couple', 'ProgramType': 'documentary', 'ProgramGenre': 'documentary', 'ProgramDuration': 'long', 'DayType': 'weekend'}),
    ('Un chico de 16 años buscando una serie de fantasía',
        {'UserAge': 'young', 'UserGender': 'male', 'ProgramType': 'series', 'ProgramGenre': 'fantasy'}),
    ('Soy una mujer de 70 años, vivo con mi hija, quiero ver las noticias por la mañana',
        {'UserAge': 'senior', 'UserGender': 'female', 'HouseholdType': 'family', 'ProgramType': 'news', 'ProgramGenre': 'news', 'TimeOfDay': 'morning'}),
    ('Somos tres amigos de unos 25 años buscando algo de humor para el viernes por la noche',
        {'UserAge': 'young', 'HouseholdType': 'family', 'ProgramGenre': 'comedy', 'DayType': 'weekend', 'TimeOfDay': 'night'}),
    ('Tengo 55 años, soy hombre y quiero algo de acción para el domingo por la tarde',
        {'UserAge': 'adult', 'UserGender': 'male', 'ProgramGenre': 'action', 'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Una chica de 28 años, soltera, buscando una comedia romántica corta',
        {'UserAge': 'young', 'UserGender': 'female', 'HouseholdType': 'single', 'ProgramGenre': 'romance', 'ProgramDuration': 'short'}),
    ('Mi marido tiene 62 años y yo 58, queremos una película de suspense esta noche',
        {'UserAge': 'senior', 'HouseholdType': 'couple', 'ProgramType': 'movie', 'ProgramGenre': 'thriller', 'TimeOfDay': 'night'}),
    ('Tengo 40 años, soy mujer, y quiero ver algo de entretenimiento el sábado por la tarde con mi familia',
        {'UserAge': 'adult', 'UserGender': 'female', 'HouseholdType': 'family', 'ProgramType': 'entertainment', 'ProgramGenre': 'entertainment', 'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Soy un señor mayor, 78 años, vivo con mi esposa y nos gusta el drama',
        {'UserAge': 'senior', 'UserGender': 'male', 'HouseholdType': 'couple', 'ProgramGenre': 'drama'}),
    ('Algo para ver antes de irme a dormir',
        {'TimeOfDay': 'night'}),
    ('Quiero ver algo después de cenar',
        {'TimeOfDay': 'night'}),
    ('Algo para la sobremesa del domingo',
        {'TimeOfDay': 'afternoon', 'DayType': 'weekend'}),
    ('Una película para el puente',
        {'DayType': 'weekend', 'ProgramType': 'movie'}),
    ('Algo para ver mientras desayuno',
        {'TimeOfDay': 'morning'}),
    ('Quiero algo para ver el fin de semana en pareja',
        {'DayType': 'weekend', 'HouseholdType': 'couple'}),
    ('Una peli corta para esta tarde con los niños',
        {'ProgramType': 'movie', 'ProgramDuration': 'short', 'TimeOfDay': 'afternoon', 'HouseholdType': 'family'}),
    ('Un jubilado buscando noticias',
        {'UserAge': 'senior', 'ProgramType': 'news', 'ProgramGenre': 'news'}),
    ('Soy universitaria y busco algo de ciencia ficción',
        {'UserAge': 'young', 'UserGender': 'female', 'ProgramGenre': 'sci-fi'}),
    ('Algo para ver en pareja el sábado noche',
        {'HouseholdType': 'couple', 'DayType': 'weekend', 'TimeOfDay': 'night'}),
    ('Otra cosa',
        {}),
    ('Dame una opción',
        {}),
    ('Algo diferente',
        {}),
    ('No me gusta eso',
        {}),
    ('Gracias',
        {}),
    ('Perfecto',
        {}),
    ('Hola, buenas tardes',
        {}),
    ('No sé qué ver',
        {}),
    ('¿Qué me recomiendas?',
        {}),
    ('Quiero ver algo',
        {}),
    ('Algo de risas pero no demasiado largo',
        {'ProgramGenre': 'comedy', 'ProgramDuration': 'short'}),
    ('Mi abuela quiere ver el telediario esta mañana',
        {'UserAge': 'senior', 'UserGender': 'female', 'ProgramType': 'news', 'ProgramGenre': 'news', 'TimeOfDay': 'morning'}),
    ('Tengo 50 y pico años',
        {'UserAge': 'adult'}),
    ('Soy treintañero',
        {'UserAge': 'adult'}),
    ('Una peli larga y de acción para el finde',
        {'ProgramType': 'movie', 'ProgramGenre': 'action', 'ProgramDuration': 'long', 'DayType': 'weekend'}),
    ('Algo de intriga policial',
        {'ProgramGenre': 'thriller'}),
    ('Una miniserie',
        {'ProgramType': 'series', 'ProgramDuration': 'short'}),
    ('Algo en blanco y negro',
        {}),
    ('Una película del oeste',
        {'ProgramType': 'movie'}),
    ('Algo de los años 80',
        {}),
    ('Una película para niños de 5 años',
        {'UserAge': 'young', 'HouseholdType': 'family'}),
    ('Quiero ver el fútbol',
        {'ProgramType': 'entertainment', 'ProgramGenre': 'entertainment'}),
    ('El partido de esta noche',
        {'TimeOfDay': 'night', 'ProgramType': 'entertainment', 'ProgramGenre': 'entertainment'}),
    ('Algo en versión original',
        {}),
    ('Un drama psicológico',
        {'ProgramGenre': 'thriller'}),
    ('Soy un chico de 17 años y quiero ver algo de acción',
        {'UserAge': 'young', 'UserGender': 'male', 'ProgramGenre': 'action'}),
    ('Tengo 42 años y vivo con mi mujer y mis tres hijos',
        {'UserAge': 'adult', 'HouseholdType': 'family'}),
    ('Una mujer de 33 años buscando una comedia',
        {'UserAge': 'young', 'UserGender': 'female', 'ProgramGenre': 'comedy'}),
    ('Estoy en casa solo un miércoles por la tarde',
        {'HouseholdType': 'single', 'DayType': 'weekday', 'TimeOfDay': 'afternoon'}),
    ('Quiero ver algo el jueves por la mañana',
        {'DayType': 'weekday', 'TimeOfDay': 'morning'}),
    ('Algo de ciencia ficción largo para el fin de semana por la noche',
        {'ProgramGenre': 'sci-fi', 'ProgramDuration': 'long', 'DayType': 'weekend', 'TimeOfDay': 'night'}),
    ('Una señorita de 24 años buscando series de romance',
        {'UserAge': 'young', 'UserGender': 'female', 'ProgramType': 'series', 'ProgramGenre': 'romance'}),
    ('Tengo 51 años, soy mujer y estoy con mi marido',
        {'UserAge': 'adult', 'UserGender': 'female', 'HouseholdType': 'couple'}),
    ('Algo de animación para ver con mis sobrinos el domingo',
        {'HouseholdType': 'family', 'DayType': 'weekend'}),
    ('Quiero ver las noticias del mediodía',
        {'ProgramType': 'news', 'ProgramGenre': 'news', 'TimeOfDay': 'afternoon'}),
    ('Un documental de naturaleza largo',
        {'ProgramType': 'documentary', 'ProgramGenre': 'documentary', 'ProgramDuration': 'long'}),
    ('Soy un hombre de 29 años, busco thriller',
        {'UserAge': 'young', 'UserGender': 'male', 'ProgramGenre': 'thriller'}),
    ('Algo para el lunes por la noche, vivo sola',
        {'DayType': 'weekday', 'TimeOfDay': 'night', 'HouseholdType': 'single'}),
    ('Una película de fantasía mediana para el sábado',
        {'ProgramType': 'movie', 'ProgramGenre': 'fantasy', 'ProgramDuration': 'medium', 'DayType': 'weekend'}),
    ('Tengo 66 años y vivo con mi pareja',
        {'UserAge': 'senior', 'HouseholdType': 'couple'}),
    ('Una serie de misterio corta entre semana',
        {'ProgramType': 'series', 'ProgramGenre': 'thriller', 'ProgramDuration': 'short', 'DayType': 'weekday'}),
    ('Somos una familia numerosa viendo algo el domingo por la tarde',
        {'HouseholdType': 'family', 'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Tengo 38 años, soy hombre y me gustan los documentales',
        {'UserAge': 'adult', 'UserGender': 'male', 'ProgramType': 'documentary', 'ProgramGenre': 'documentary'}),
    ('Algo de entretenimiento para el viernes por la noche',
        {'ProgramType': 'entertainment', 'ProgramGenre': 'entertainment', 'DayType': 'weekend', 'TimeOfDay': 'night'}),
    ('Una chica de 22 años buscando algo de horror corto',
        {'UserAge': 'young', 'UserGender': 'female', 'ProgramGenre': 'horror', 'ProgramDuration': 'short'}),
    ('Quiero ver algo de drama esta tarde en pareja',
        {'ProgramGenre': 'drama', 'TimeOfDay': 'afternoon', 'HouseholdType': 'couple'}),
    ('Soy un hombre mayor de 71 años viendo las noticias por la mañana',
        {'UserAge': 'senior', 'UserGender': 'male', 'ProgramType': 'news', 'ProgramGenre': 'news', 'TimeOfDay': 'morning'}),
    ('Algo ligero para el martes por la tarde',
        {'DayType': 'weekday', 'TimeOfDay': 'afternoon'}),
    ('Una película de acción larga para el sábado noche con mi novio',
        {'ProgramType': 'movie', 'ProgramGenre': 'action', 'ProgramDuration': 'long', 'DayType': 'weekend', 'TimeOfDay': 'night', 'HouseholdType': 'couple'}),
    ('Tengo 49 años y busco una comedia corta para el miércoles',
        {'UserAge': 'adult', 'ProgramGenre': 'comedy', 'ProgramDuration': 'short', 'DayType': 'weekday'}),
    ('Soy estudiante de 21 años, vivo solo y busco sci-fi',
        {'UserAge': 'young', 'HouseholdType': 'single', 'ProgramGenre': 'sci-fi'}),
    ('Una película de romance para el domingo tarde',
        {'ProgramType': 'movie', 'ProgramGenre': 'romance', 'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Tengo 77 años y vivo con mi hijo',
        {'UserAge': 'senior', 'HouseholdType': 'family'}),
    ('Algo de humor para el sábado tarde entre semana',
        {'ProgramGenre': 'comedy', 'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Una señora de 48 años sola en casa el viernes',
        {'UserAge': 'adult', 'UserGender': 'female', 'HouseholdType': 'single', 'DayType': 'weekend'}),
    ('Algo con niños pequeños en casa, mañana de sábado',
        {'HouseholdType': 'family', 'DayType': 'weekend', 'TimeOfDay': 'morning'}),
    ('Quiero algo de fantasía épica larga',
        {'ProgramGenre': 'fantasy', 'ProgramDuration': 'long'}),
    ('Somos dos chicas de 26 años buscando algo de romance',
        {'UserAge': 'young', 'UserGender': 'female', 'HouseholdType': 'couple', 'ProgramGenre': 'romance'}),
    ('Un hombre de 56 años buscando un thriller psicológico',
        {'UserAge': 'senior', 'UserGender': 'male', 'ProgramGenre': 'thriller'}),
    ('Algo para ver el lunes por la mañana solita',
        {'DayType': 'weekday', 'TimeOfDay': 'morning', 'HouseholdType': 'single'}),
    ('Tengo 33 años, estoy con mi marido y quiero una serie de drama larga',
        {'UserAge': 'young', 'HouseholdType': 'couple', 'ProgramType': 'series', 'ProgramGenre': 'drama', 'ProgramDuration': 'long'}),
    ('Soy una mujer de 62 años buscando documentales por la tarde',
        {'UserAge': 'senior', 'UserGender': 'female', 'ProgramType': 'documentary', 'ProgramGenre': 'documentary', 'TimeOfDay': 'afternoon'}),
    ('Algo de ciencia ficción corto para entre semana',
        {'ProgramGenre': 'sci-fi', 'ProgramDuration': 'short', 'DayType': 'weekday'}),
    ('Quiero una comedia familiar para el domingo mediodía',
        {'ProgramGenre': 'comedy', 'HouseholdType': 'family', 'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Tengo 44 años, vivo con mis dos hijos y quiero algo de acción el sábado',
        {'UserAge': 'adult', 'HouseholdType': 'family', 'ProgramGenre': 'action', 'DayType': 'weekend'}),
    ('Una chica soltera de 31 años buscando thriller corto',
        {'UserAge': 'young', 'UserGender': 'female', 'HouseholdType': 'single', 'ProgramGenre': 'thriller', 'ProgramDuration': 'short'}),
    ('Algo de entretenimiento para toda la familia el domingo tarde',
        {'ProgramType': 'entertainment', 'ProgramGenre': 'entertainment', 'HouseholdType': 'family', 'DayType': 'weekend', 'TimeOfDay': 'afternoon'}),
    ('Tengo 59 años y busco una película de romance larga',
        {'UserAge': 'adult', 'ProgramType': 'movie', 'ProgramGenre': 'romance', 'ProgramDuration': 'long'}),
    ('Soy un jubilado de 69 años viendo series por las mañanas',
        {'UserAge': 'senior', 'UserGender': 'male', 'ProgramType': 'series', 'TimeOfDay': 'morning'}),
    ('Algo de drama histórico para el fin de semana',
        {'ProgramGenre': 'drama', 'DayType': 'weekend'}),
    ('Una pareja de 40 años buscando algo de acción un viernes noche',
        {'UserAge': 'adult', 'HouseholdType': 'couple', 'ProgramGenre': 'action', 'DayType': 'weekend', 'TimeOfDay': 'night'}),
    ('Soy una abuela de 73 años, me gustan las noticias y los documentales',
        {'UserAge': 'senior', 'UserGender': 'female'}),
    ('Algo corto para ver antes de trabajar',
        {'ProgramDuration': 'short', 'TimeOfDay': 'morning'}),
    ('Una película de terror para el sábado noche solo',
        {'ProgramType': 'movie', 'ProgramGenre': 'horror', 'DayType': 'weekend', 'TimeOfDay': 'night', 'HouseholdType': 'single'}),
    ('Tengo 28 años y busco una serie de ciencia ficción',
        {'UserAge': 'young', 'ProgramType': 'series', 'ProgramGenre': 'sci-fi'}),
    ('Una película corta para la sobremesa',
        {'ProgramType': 'movie', 'ProgramDuration': 'short', 'TimeOfDay': 'afternoon'}),
    ('Tengo 55 años y prefiero películas largas de drama',
        {'UserAge': 'adult', 'ProgramType': 'movie', 'ProgramGenre': 'drama', 'ProgramDuration': 'long'}),
    ('Una chica joven de 22 años buscando series de romance',
        {'UserAge': 'young', 'UserGender': 'female', 'ProgramType': 'series', 'ProgramGenre': 'romance'}),
    ('Quiero ver un documental el jueves por la noche',
        {'ProgramType': 'documentary', 'ProgramGenre': 'documentary', 'DayType': 'weekday', 'TimeOfDay': 'night'}),
    ('Soy un hombre de 45 años buscando algo de acción',
        {'UserAge': 'adult', 'UserGender': 'male', 'ProgramGenre': 'action'}),
    ('Algo corto y entretenido para ver en pareja',
        {'ProgramDuration': 'short', 'HouseholdType': 'couple'}),
    ('Una familia con niños buscando algo el sábado por la mañana',
        {'HouseholdType': 'family', 'DayType': 'weekend', 'TimeOfDay': 'morning'}),
    ('Tengo 38 años y busco una película de thriller el viernes',
        {'UserAge': 'adult', 'ProgramType': 'movie', 'ProgramGenre': 'thriller', 'DayType': 'weekend'}),
    ('Soy una mujer de 50 años y me gustan las series largas de drama',
        {'UserAge': 'adult', 'UserGender': 'female', 'ProgramType': 'series', 'ProgramGenre': 'drama', 'ProgramDuration': 'long'}),
    ('Quiero ver algo el domingo por la mañana tranquilo',
        {'DayType': 'weekend', 'TimeOfDay': 'morning'}),
    ('Un hombre mayor de 70 años que busca películas',
        {'UserAge': 'senior', 'UserGender': 'male', 'ProgramType': 'movie'}),
    ('Somos una pareja joven buscando una comedia esta noche',
        {'HouseholdType': 'couple', 'ProgramGenre': 'comedy', 'TimeOfDay': 'night'}),
    ('Tengo 26 años, estoy solo y quiero ver un thriller largo',
        {'UserAge': 'young', 'HouseholdType': 'single', 'ProgramGenre': 'thriller', 'ProgramDuration': 'long'}),
    ('Una serie de ciencia ficción larga para el fin de semana',
        {'ProgramType': 'series', 'ProgramGenre': 'sci-fi', 'ProgramDuration': 'long', 'DayType': 'weekend'}),
    ('Busco algo de fantasía para ver con mis hijos el sábado',
        {'ProgramGenre': 'fantasy', 'HouseholdType': 'family', 'DayType': 'weekend'}),
    ('Soy una señora de 48 años y me apetece ver noticias por la mañana',
        {'UserAge': 'adult', 'UserGender': 'female', 'ProgramGenre': 'news', 'TimeOfDay': 'morning'}),
    ('Quiero algo de entretenimiento ligero para el mediodía',
        {'ProgramGenre': 'entertainment', 'TimeOfDay': 'afternoon'}),
]

print(f"Intent test cases:     {len(INTENT_TEST_CASES)}")
print(f"Extraction test cases: {len(EXTRACTION_TEST_CASES)}")

In [ ]:
models = _agent.client.models.list()
for m in models:
    print(m.id)


#### Data Structures and Metric Helpers

In [ ]:
@dataclass
class IntentResult:
    message:    str
    expected:   str
    predicted:  str
    correct:    bool
    difficulty: str
    notes:      str
    latency_ms: float
    tokens_in:  int
    tokens_out: int


@dataclass
class ExtractionResult:
    message:       str
    expected:      dict
    predicted:     dict
    field_hits:    int
    field_misses:  int
    hallucinations:int
    invalid_values:int
    latency_ms:    float
    tokens_in:     int
    tokens_out:    int


# ── Metric helpers ────────────────────────────────────────────────────────────

def confusion_matrix_data(results, classes):
    idx = {c: i for i, c in enumerate(classes)}
    n   = len(classes)
    mat = [[0] * n for _ in range(n)]
    for r in results:
        if r.expected in idx and r.predicted in idx:
            mat[idx[r.expected]][idx[r.predicted]] += 1
        elif r.expected in idx:
            mat[idx[r.expected]][idx.get("OTHER", 0)] += 1
    return mat


def per_class_metrics(results, classes):
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
    for r in results:
        if r.predicted == r.expected:
            tp[r.expected] += 1
        else:
            fp[r.predicted] += 1
            fn[r.expected]  += 1
    metrics = {}
    for c in classes:
        p  = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0.0
        r_ = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0.0
        f1 = 2 * p * r_ / (p + r_) if (p + r_) > 0 else 0.0
        metrics[c] = {"precision": p, "recall": r_, "f1": f1, "support": tp[c] + fn[c]}
    return metrics


def macro_avg(per_class):
    keys = list(per_class)
    return {k: sum(per_class[c][k] for c in keys) / len(keys) for k in ("precision","recall","f1")}


def weighted_avg(per_class):
    total = sum(per_class[k]["support"] for k in per_class)
    if not total:
        return {"precision": 0, "recall": 0, "f1": 0}
    return {k: sum(per_class[c][k] * per_class[c]["support"] for c in per_class) / total
            for k in ("precision","recall","f1")}


def latency_stats(values):
    s = sorted(values); n = len(s)
    pct = lambda p: s[int(p/100*(n-1))] if n else 0
    return {"mean_ms": sum(s)/n if n else 0,
            "p50_ms": pct(50), "p95_ms": pct(95), "max_ms": s[-1] if s else 0}

print("✅ Helpers defined")

#### Intent Classification Tests

In [ ]:
def run_intent_tests(verbose=True):
    results = []
    for message, expected, difficulty, notes in INTENT_TEST_CASES:
        t0 = time.perf_counter()
        try:
            resp = openai_client.chat.completions.create(
                model=INT_MODEL,
                messages=[
                    {"role": "system", "content": _agent.INTENT_PROMPT},
                    {"role": "user",   "content": message},
                ],
                temperature=0,
            )
            latency_ms = (time.perf_counter() - t0) * 1000
            content    = _agent.clean_json_response(resp.choices[0].message.content)
            try:
                predicted = json.loads(content).get("intent", "OTHER")
            except json.JSONDecodeError:
                predicted = "OTHER"
            tokens_in  = resp.usage.prompt_tokens
            tokens_out = resp.usage.completion_tokens
        except Exception as e:
            latency_ms = (time.perf_counter() - t0) * 1000
            predicted  = "OTHER"
            tokens_in = tokens_out = 0
            print(f"  ❌ API error: {e}")

        correct = predicted == expected
        results.append(IntentResult(
            message=message, expected=expected, predicted=predicted,
            correct=correct, difficulty=difficulty, notes=notes,
            latency_ms=latency_ms, tokens_in=tokens_in, tokens_out=tokens_out,
        ))
        if verbose:
            icon = "✅" if correct else "❌"
            diff_tag = f"[{difficulty}]"
            print(f"  {icon} {diff_tag:<8} {message:<44} | expected: {expected:<15} | got: {predicted:<15} | {latency_ms:5.0f}ms")

    classes   = sorted(VALID_INTENTS)
    pc        = per_class_metrics(results, classes)
    correct_n = sum(1 for r in results if r.correct)
    accuracy  = correct_n / len(results)
    lat       = latency_stats([r.latency_ms for r in results])
    total_in  = sum(r.tokens_in  for r in results)
    total_out = sum(r.tokens_out for r in results)

    diff_groups = defaultdict(list)
    for r in results:
        diff_groups[r.difficulty].append(r.correct)
    diff_acc = {d: sum(v)/len(v) for d, v in diff_groups.items()}

    summary = {
        "accuracy": accuracy, "correct": correct_n, "total": len(results),
        "per_class": pc, "macro_avg": macro_avg(pc), "weighted_avg": weighted_avg(pc),
        "by_difficulty": diff_acc, "latency": lat,
        "confusion_matrix": {"classes": classes, "matrix": confusion_matrix_data(results, classes)},
    }
    return results, summary

intent_results, intent_summary = run_intent_tests(verbose=True)
print(f"\nIntent tests complete — Accuracy: {intent_summary['accuracy']:.1%}")

#### Attribute Extraction Tests

In [ ]:
def run_extraction_tests(verbose=True):
    results = []
    for message, expected_attrs in EXTRACTION_TEST_CASES:
        t0 = time.perf_counter()
        try:
            resp = mo_client.chat.completions.create(
                model=ATT_MODEL,
                messages=[
                    {"role": "system", "content": _agent.EXTRACTION_PROMPT},
                    {"role": "user",   "content": message},
                ],
                temperature=0,
            )
            latency_ms = (time.perf_counter() - t0) * 1000
            content    = _agent.clean_json_response(resp.choices[0].message.content)
            try:
                predicted_raw = json.loads(content)
            except json.JSONDecodeError:
                predicted_raw = {}
            tokens_in  = resp.usage.prompt_tokens
            tokens_out = resp.usage.completion_tokens
        except Exception as e:
            latency_ms = (time.perf_counter() - t0) * 1000
            predicted_raw = {}
            tokens_in = tokens_out = 0
            print(f"  ❌ API error: {e}")

        predicted = {k: (None if v in (None, "", "null", "NULL") else v)
                     for k, v in predicted_raw.items()}

        hits = misses = hallucinations = invalid_values = 0
        field_details = []

        for key in VALID_ATTRIBUTES:
            exp_val  = expected_attrs.get(key)
            pred_val = predicted.get(key)
            if exp_val is None:
                if pred_val is not None:
                    hallucinations += 1
                    field_details.append(("HALLUCINATION", key, None, pred_val))
            else:
                if pred_val == exp_val:
                    hits += 1
                    field_details.append(("OK", key, exp_val, pred_val))
                elif pred_val is None:
                    misses += 1
                    field_details.append(("MISS", key, exp_val, pred_val))
                elif pred_val not in VALID_ATTRIBUTES.get(key, set()):
                    invalid_values += 1; misses += 1
                    field_details.append(("INVALID", key, exp_val, pred_val))
                else:
                    misses += 1
                    field_details.append(("WRONG", key, exp_val, pred_val))

        results.append(ExtractionResult(
            message=message, expected=expected_attrs, predicted=predicted,
            field_hits=hits, field_misses=misses,
            hallucinations=hallucinations, invalid_values=invalid_values,
            latency_ms=latency_ms, tokens_in=tokens_in, tokens_out=tokens_out,
        ))

        if verbose:
            ok = misses == 0 and hallucinations == 0
            icon = "✅" if ok else "❌"
            exp_n = len(expected_attrs)
            print(f"\n  {icon} \"{message}\"")
            print(f"     hits={hits}/{exp_n}  misses={misses}  hallucinations={hallucinations}  invalid={invalid_values}  {latency_ms:.0f}ms")
            for tag, key, exp_v, pred_v in field_details:
                if tag == "OK":
                    print(f"       ✔ {key}: '{pred_v}'")
                elif tag == "HALLUCINATION":
                    print(f"       🔴 HALLUCINATION {key}: expected null → got '{pred_v}'")
                elif tag == "MISS":
                    print(f"       🟡 MISS {key}: expected '{exp_v}' → got null")
                elif tag == "WRONG":
                    print(f"       🟠 WRONG {key}: expected '{exp_v}' → got '{pred_v}'")
                elif tag == "INVALID":
                    print(f"       🔴 INVALID {key}: expected '{exp_v}' → got '{pred_v}' (not in allowed set)")

    total_expected = sum(len(r.expected) for r in results)
    total_hits     = sum(r.field_hits for r in results)
    total_hallucs  = sum(r.hallucinations for r in results)
    total_invalid  = sum(r.invalid_values for r in results)
    perfect_cases  = sum(1 for r in results if r.field_misses == 0 and r.hallucinations == 0)
    lat            = latency_stats([r.latency_ms for r in results])
    total_in       = sum(r.tokens_in  for r in results)
    total_out      = sum(r.tokens_out for r in results)

    field_hits_map = defaultdict(list)
    for r in results:
        for key in VALID_ATTRIBUTES:
            exp_val  = r.expected.get(key)
            pred_val = r.predicted.get(key)
            if exp_val is None:
                field_hits_map[key].append(pred_val is None)
            else:
                field_hits_map[key].append(pred_val == exp_val)
    per_field_accuracy = {k: sum(v)/len(v) for k, v in field_hits_map.items()}

    summary = {
        "perfect_cases": perfect_cases, "total_cases": len(results),
        "perfect_rate": perfect_cases / len(results),
        "field_hit_rate": total_hits / total_expected if total_expected else 0,
        "total_hallucinations": total_hallucs,
        "total_invalid_values": total_invalid,
        "per_field_accuracy": per_field_accuracy,
        "latency": lat,
    }
    return results, summary

extraction_results, extraction_summary = run_extraction_tests(verbose=True)
print(f"\n✅ Extraction tests complete — Perfect cases: {extraction_summary['perfect_rate']:.1%}")

#### Intent Classification Results

In [ ]:
# ── Per-class metrics table ───────────────────────────────────────────────────
pc = intent_summary["per_class"]
df_intent = pd.DataFrame([
    {"Class": c, "Precision": m["precision"], "Recall": m["recall"], "F1": m["f1"], "Support": m["support"]}
    for c, m in sorted(pc.items())
])

macro   = intent_summary["macro_avg"]
weighted_= intent_summary["weighted_avg"]
df_avgs  = pd.DataFrame([
    {"Class": "macro avg",    "Precision": macro["precision"],    "Recall": macro["recall"],    "F1": macro["f1"],    "Support": ""},
    {"Class": "weighted avg", "Precision": weighted_["precision"],"Recall": weighted_["recall"],"F1": weighted_["f1"],"Support": ""},
])
df_all = pd.concat([df_intent, df_avgs], ignore_index=True)


styled = (df_all.style
    .format({"Precision": "{:.2f}", "Recall": "{:.2f}", "F1": "{:.2f}"}, na_rep="")
    .set_caption(f"Overall Accuracy: {intent_summary['accuracy']:.1%}  "
                 f"({intent_summary['correct']}/{intent_summary['total']} correct)"))
display(styled)

In [ ]:
# ── Confusion matrix heatmap ─────────────────────────────────────────────────
import numpy as np

cm   = intent_summary["confusion_matrix"]
mat  = cm["matrix"]
cls  = cm["classes"]
mat_arr = np.array(mat)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(mat_arr, annot=True, fmt="d", cmap="Greens",
            xticklabels=cls, yticklabels=cls,
            linewidths=0.5, linecolor="grey", ax=ax)

# Highlight off-diagonal non-zero cells in red
for i in range(len(cls)):
    for j in range(len(cls)):
        if i != j and mat_arr[i, j] > 0:
            ax.add_patch(plt.Rectangle((j, i), 1, 1,
                fill=True, facecolor="red", alpha=0.35,
                edgecolor="red", linewidth=2))

# small-talk(expected) and other (predicted) cell highlight in orange
if "SMALLTALK" in cls and "OTHER" in cls:
    i = cls.index("SMALLTALK")
    j = cls.index("OTHER")
    if mat_arr[i, j] > 0:
        ax.add_patch(plt.Rectangle((j, i), 1, 1,
            fill=True, facecolor="yellow", alpha=0.35,
            edgecolor="orange", linewidth=2))
        
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Expected",  fontsize=12)
ax.set_title("Intent Classification Matrix", fontsize=14, fontweight="bold")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ── Accuracy by difficulty ───────────────────────────────────────────────────
diff_data = intent_summary["by_difficulty"]
diff_order = ["easy", "medium", "hard"]
diff_vals  = [diff_data.get(d, 0) for d in diff_order]

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(diff_order, diff_vals,
              color=["#28a745" if v >= 0.90 else "#ffc107" if v >= 0.70 else "#dc3545"
                     for v in diff_vals],
              edgecolor="white", width=0.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Accuracy")
ax.set_title("Intent Accuracy by Difficulty")
for bar, val in zip(bars, diff_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.0%}", ha="center", fontsize=11, fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


#### Attribute Extraction Results

In [ ]:
# ── Summary KPIs ─────────────────────────────────────────────────────────────
es = extraction_summary
print(f"  Perfect cases  : {es['perfect_cases']}/{es['total_cases']}  ({es['perfect_rate']:.1%})")
print(f"  Field hit rate : {es['field_hit_rate']:.1%}")
print(f"  Hallucinations : {es['total_hallucinations']}")
print(f"  Invalid values : {es['total_invalid_values']}")


In [ ]:
# ── Per-field accuracy bar chart ─────────────────────────────────────────────
pfa   = extraction_summary["per_field_accuracy"]
fields = sorted(pfa, key=pfa.get, reverse=True)
vals   = [pfa[f] for f in fields]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(fields, vals,
               color=["#28a745" if v >= 0.9 else "#ffc107" if v >= 0.7 else "#dc3545"
                      for v in vals],
               edgecolor="white")
ax.set_xlim(0, 1.15)
ax.set_xlabel("Accuracy")
ax.set_title("Per-field Extraction Accuracy", fontsize=13, fontweight="bold")
ax.axvline(1.0, color="grey", linestyle="--", linewidth=0.7)
for bar, val in zip(bars, vals):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f"{val:.0%}", va="center", fontsize=10)
plt.tight_layout()
plt.show()


#### Latency

In [ ]:
# ── Latency comparison table ─────────────────────────────────────────────────
rows = []
for label, summary in [("Intent", intent_summary), ("Extraction", extraction_summary)]:
    lat = summary["latency"]
    rows.append({
        "Task":       label,
        "Mean (ms)":  round(lat["mean_ms"]),
        "p50 (ms)":   round(lat["p50_ms"]),
        "p95 (ms)":   round(lat["p95_ms"]),
        "Max (ms)":   round(lat["max_ms"]),
    })

display(pd.DataFrame(rows).set_index("Task"))

In [ ]:
# ── Per-call latency distribution ────────────────────────────────────────────
intent_lats     = [r.latency_ms for r in intent_results]
extraction_lats = [r.latency_ms for r in extraction_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lats, title in zip(
    axes,
    [intent_lats, extraction_lats],
    ["Intent Classification — Latency (ms)", "Attribute Extraction — Latency (ms)"]
):
    ax.hist(lats, bins=12, color="#4eb054", edgecolor="white", alpha=0.85)
    ax.axvline(pd.Series(lats).median(), color="orange", linestyle="--", linewidth=1.5, label="p50")
    ax.axvline(pd.Series(lats).quantile(0.95), color="red", linestyle="--", linewidth=1.5, label="p95")
    ax.set_xlabel("ms"); ax.set_ylabel("Count")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


#### Summary & Actionable Insights

In [ ]:
ia         = intent_summary["accuracy"]
iw         = intent_summary["weighted_avg"]["f1"]
er         = extraction_summary["perfect_rate"]
eh         = extraction_summary["total_hallucinations"]

print("=" * 56)
print("  SUMMARY")
print("=" * 56)
print(f"  Intent Classification")
print(f"    Accuracy         {ia:.1%}")
print(f"    Weighted F1      {iw:.1%}")
print()
print(f"  Attribute Extraction")
print(f"    Perfect cases    {er:.1%}")
print(f"    Hallucinations   {eh}")
print()
print("=" * 56)

print("\n  Actionable Insights:")
worst_classes = sorted(intent_summary["per_class"].items(), key=lambda x: x[1]["f1"])[:2]
for cls, m in worst_classes:
    if m["support"] > 0 and m["f1"] < 0.85:
        print(f"  ⚠️  Intent '{cls}' has low F1={m['f1']:.2f}")

if eh > 0:
    print(f"  ⚠️  {eh} hallucination(s) in extraction")

if intent_summary["by_difficulty"].get("hard", 1.0) < 0.7:
    print("  ⚠️  Hard cases underperform")

if ia >= 0.92 and er >= 0.85:
    print("  ✅ Both prompts performing well.")

#### Save Full Results to JSON

In [ ]:
import json, pathlib

out = {"intent": intent_summary, "extraction": extraction_summary}
path = pathlib.Path("output/llm_eval_results.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False, default=str)

print(f"✅ Results saved to {path.resolve()}")
